# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mr-PeterMaged/flyrank/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two signal checks, then my rule and its reason codes

Same lane as ML-04 (Refresh / Content Opportunity Scoring), same decision point
(`2026-03-15`, mid-panel March), same honest-features discipline: everything below is
knowable by `2026-03-15` — no `_sample` table, no second-half data, no `is_declining_label`
from w03.

**Two signals my rule idea leans on:**
1. **Staleness (content age)** -- behind FlyRank's refresh flags. Hypothesis: older content
   shows lower CTR.
2. **CTR-vs-position** -- behind FlyRank's CTR-fix logic. Hypothesis: CTR falls as position
   gets worse.

I check both with a bucket table before coding anything.

In [1]:
import os
import getpass
import duckdb
import pandas as pd
import numpy as np

pd.set_option("display.width", 160)

# Token order: env var -> Colab Secret -> prompt (last resort). Never paste into a cell.
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf_token (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
MARCH = f"{REL}/fact_content_daily_performance/month=2026-03/data_0.parquet"
DIM_CONTENT = f"{REL}/dim_content.parquet"
DECISION_DATE = pd.Timestamp("2026-03-15")

# Same first-half base as w03 (report_date 2026-03-01..15, gsc_data_available IS TRUE),
# joined to dim_content for content_created_date -- the ONLY dim_content date field that
# is safe to use as of the decision point (checked below).
base = con.execute(f"""
    WITH fh AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions) AS impressions_first_half,
               SUM(gsc_clicks) AS clicks_first_half,
               AVG(gsc_avg_position) AS avg_position_first_half
        FROM '{MARCH}'
        WHERE gsc_data_available IS TRUE
          AND report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT fh.*, dc.content_created_date, dc.content_updated_date
    FROM fh
    JOIN '{DIM_CONTENT}' dc USING (client_hash_id, content_hash_id)
    WHERE fh.impressions_first_half > 0
      AND dc.is_published IS TRUE AND dc.is_deleted IS FALSE
      AND dc.content_created_date <= DATE '2026-03-15'
""").df()

base["ctr_first_half"] = base["clicks_first_half"] / base["impressions_first_half"]
base["content_age_days"] = (DECISION_DATE - pd.to_datetime(base["content_created_date"])).dt.days

leak_share = (pd.to_datetime(base["content_updated_date"]) > DECISION_DATE).mean()
print("base rows:", len(base))
print(f"content_updated_date is AFTER the decision date for {leak_share:.1%} of rows -- "
      f"using it as 'days since update' would leak the future. Using content_created_date "
      f"(content_age_days) instead: it is a one-time fact, never revised.")

base rows: 151837
content_updated_date is AFTER the decision date for 82.7% of rows -- using it as 'days since update' would leak the future. Using content_created_date (content_age_days) instead: it is a one-time fact, never revised.


**Signal 1 -- staleness (content age vs CTR).** Claim: "older content gets lower CTR."
Buckets by `content_age_days`, weighted CTR (`sum(clicks) / sum(impressions)` per bucket, not
the mean of per-page rates -- avoids the averaging trap).

In [2]:
def stale_tier(d):
    if d < 90: return "<90d"
    if d < 180: return "90-179d"
    if d < 365: return "180-364d"
    return "365d+"

base["staleness_tier"] = base["content_age_days"].apply(stale_tier)
tier_order = ["<90d", "90-179d", "180-364d", "365d+"]

signal_a = base.groupby("staleness_tier").agg(
    n=("ctr_first_half", "size"),
    total_clicks=("clicks_first_half", "sum"),
    total_impressions=("impressions_first_half", "sum"),
).reindex(tier_order)
signal_a["weighted_ctr"] = signal_a["total_clicks"] / signal_a["total_impressions"]
print("Signal 1 -- content age vs CTR (n >= 50 floor met in every bucket):")
print(signal_a)

Signal 1 -- content age vs CTR (n >= 50 floor met in every bucket):
                    n  total_clicks  total_impressions  weighted_ctr
staleness_tier                                                      
<90d            48848      128579.0         38433472.0      0.003345
90-179d         29161       79778.0         29948678.0      0.002664
180-364d        58693      144383.0         48703179.0      0.002965
365d+           15135       32002.0         10403794.0      0.003076


**Verdict: MIXED.** Weighted CTR by age bucket -- `<90d: 0.335%`, `90-179d: 0.266%`,
`180-364d: 0.297%`, `365d+: 0.308%` -- is NOT monotonic: it dips at 90-179 days, then climbs
back up, and the oldest bucket (365d+) actually lands close to the newest bucket rather than
below it. All four buckets clear the sample-size floor (15k-59k rows each), so this isn't a
tiny-n fluke -- the relationship genuinely doesn't hold in a clean, usable way. **This
negative result is the useful outcome of the check: content age will NOT drive my rule's
score.** (`content_updated_date` -- the more direct "days since last touched" field -- can't
be used at all: it postdates the decision point for 82.7% of rows, so it isn't knowable at
decision time.)

**Signal 2 -- CTR vs. position** (behind FlyRank's CTR-fix logic). Claim: "CTR falls as
position gets worse."

In [3]:
def pos_tier(p):
    if p <= 0: return "no_rank"
    if p <= 3: return "1-3"
    if p <= 10: return "4-10"
    if p <= 20: return "11-20"
    return "21+"

base["position_tier"] = base["avg_position_first_half"].apply(pos_tier)
pos_order = ["1-3", "4-10", "11-20", "21+", "no_rank"]

signal_b = base.groupby("position_tier").agg(
    n=("ctr_first_half", "size"),
    total_clicks=("clicks_first_half", "sum"),
    total_impressions=("impressions_first_half", "sum"),
).reindex(pos_order)
signal_b["weighted_ctr"] = signal_b["total_clicks"] / signal_b["total_impressions"]
print("Signal 2 -- position vs CTR:")
print(signal_b)

Signal 2 -- position vs CTR:
                   n  total_clicks  total_impressions  weighted_ctr
position_tier                                                      
1-3            16238       89755.0         19932559.0      0.004503
4-10           69885      202392.0         61645528.0      0.003283
11-20          26904       50686.0         15013423.0      0.003376
21+            37506       41863.0         30894872.0      0.001355
no_rank         1304          46.0             2741.0      0.016782


**Verdict: CONFIRMED.** Weighted CTR falls with position as expected: `1-3: 0.450%` ->
`4-10: 0.328%` -> `11-20: 0.338%` -> `21+: 0.136%`. The main trend is clean (top-3 pages get
roughly 3.3x the CTR of position-21+ pages); the one wrinkle is `4-10` and `11-20` sitting
almost on top of each other (0.328% vs 0.338%, a ~3% relative gap -- noise-level, not a real
inversion). `no_rank` (n=1,304, avg_position recorded as 0/missing) is excluded from the
trend read -- it's a tiny, structurally different bucket (near-zero impressions per page) and
reading it as "high CTR" would be exactly the kind of small-n noise the floor rule exists to
catch.

**This is the signal my rule uses** -- and it's also the flag-linked one: this is the same
CTR-vs-position relationship behind FlyRank's CTR-fix logic from the session.

**My rule, in plain words:** *"A page is worth a CTR review if it's visible, ranks in the
top 20, and its click-through rate is below the median CTR of other visible pages at the same
position tier."* One score, one reason code (`low_ctr_visible_page`), one action label
(`review_for_ctr_fix`). No content age (signal 1 said no), no `is_declining_label`, no
second-half data -- only signal 2, which held up.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [4]:
# Visible + ranked universe: has a real position (top 20) and enough impressions to trust
# a CTR estimate. 200 impressions is close to the 60th percentile of this subset -- round,
# defensible, not fit to the data.
visible = base[(base["avg_position_first_half"] > 0)
               & (base["avg_position_first_half"] <= 20)
               & (base["impressions_first_half"] >= 200)].copy()

peer_median_ctr = visible.groupby("position_tier")["ctr_first_half"].median()
print("peer median CTR per position tier (the 'expected' CTR a page is compared against):")
print(peer_median_ctr)

visible["peer_median_ctr"] = visible["position_tier"].map(peer_median_ctr)
visible["low_ctr_flag"] = visible["ctr_first_half"] < visible["peer_median_ctr"]

# score = estimated missed clicks if this page matched its peer group's median CTR --
# readable on purpose, no fitted weights.
visible["score"] = (
    (visible["peer_median_ctr"] - visible["ctr_first_half"]).clip(lower=0)
    * visible["impressions_first_half"]
).where(visible["low_ctr_flag"], 0.0)

visible["reason_code"] = np.where(visible["low_ctr_flag"], "low_ctr_visible_page", "none")
visible["action_label"] = np.where(visible["low_ctr_flag"], "review_for_ctr_fix", "monitor")

queue = visible.sort_values(["score", "content_hash_id"], ascending=[False, True]).reset_index(drop=True)
queue["rank"] = queue.index + 1

print(f"\nqueue rows: {len(queue)}  |  flagged (score > 0): {(queue['score'] > 0).sum()}")
print(f"top score: {queue['score'].max():.1f}  |  median score among flagged: "
      f"{queue.loc[queue['score'] > 0, 'score'].median():.1f}")

out_cols = ["rank", "client_hash_id", "content_hash_id", "position_tier",
            "avg_position_first_half", "impressions_first_half", "clicks_first_half",
            "ctr_first_half", "peer_median_ctr", "score", "reason_code", "action_label"]
out_path = "../outputs/baseline_action_score.csv"
os.makedirs("../outputs", exist_ok=True)
queue[out_cols].to_csv(out_path, index=False)
print(f"\nwrote {out_path}")

peer median CTR per position tier (the 'expected' CTR a page is compared against):
position_tier
1-3      0.002783
11-20    0.001860
4-10     0.002101
Name: ctr_first_half, dtype: float64

queue rows: 50749  |  flagged (score > 0): 25370
top score: 186.9  |  median score among flagged: 1.0



wrote ../outputs/baseline_action_score.csv


**Queue stats:** 50,749 visible, top-20-ranked pages scored; **25,370 (50.0%)** are flagged
`low_ctr_visible_page`, the rest are `monitor` (score 0, sit at the bottom of the ranking).
Peer median CTR is highest for `1-3` (0.278%), lower for `4-10` (0.210%) and `11-20`
(0.186%) -- matching the position curve confirmed in signal 2, which is exactly why using a
*per-tier* median (not one global median) matters: comparing an 11-20 page against the 1-3
median would flag almost everyone at the bottom of page one, for no real reason.

## 3. Top-10 review

For each of the top 10: the action, why it's there, and what would make it wrong.

In [5]:
top10 = queue.head(10)[["rank", "client_hash_id", "position_tier", "avg_position_first_half",
                          "impressions_first_half", "clicks_first_half", "ctr_first_half",
                          "peer_median_ctr", "score"]]
top10.insert(1, "client", top10["client_hash_id"].str.slice(7, 15))  # short id for display only
top10 = top10.drop(columns="client_hash_id")
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
print(top10.to_string(index=False))

 rank   client position_tier  avg_position_first_half  impressions_first_half  clicks_first_half  ctr_first_half  peer_median_ctr    score
    1 62f4a7e6           1-3                   2.7867             73,639.0000            18.0000          0.0002           0.0028 186.9323
    2 73cda7b4          4-10                   8.6079             83,772.0000             0.0000          0.0000           0.0021 175.9916
    3 62f4a7e6          4-10                   5.7855             86,860.0000            51.0000          0.0006           0.0021 131.4790
    4 73cda7b4          4-10                   4.5790             58,553.0000             1.0000          0.0000           0.0021 122.0105
    5 23a62021          4-10                   9.0135             55,680.0000            15.0000          0.0003           0.0021 101.9748
    6 62f4a7e6          4-10                   6.4138             49,314.0000             2.0000          0.0000           0.0021 101.6008
    7 20259bd6         11-2

**Top-10, one line each:**

1. **client 62f4a7e6, pos 2.8 (tier 1-3), 73,639 impr, 18 clicks (CTR 0.024%).** Action:
   `review_for_ctr_fix`. Why: ranks top-3 but CTR is 1/11th of the tier-median (2.78%). What
   would make it wrong: if this URL is mid-migration or has a pending canonical/redirect --
   check GSC's "why pages aren't indexed" report before touching the title.
2. **client 73cda7b4, pos 8.6 (tier 4-10), 83,772 impr, 0 clicks.** Action:
   `review_for_ctr_fix`. Why: highest score in the queue -- huge visibility, literally zero
   clicks. What would make it wrong: zero clicks on 83k impressions looks more like a
   tracking/attribution gap (e.g. GSC counting impressions for a URL that redirects
   client-side) than a title/snippet problem -- verify clicks are really zero in GA4 too
   before writing new copy.
3. **client 62f4a7e6, pos 5.8 (tier 4-10), 86,860 impr, 51 clicks (CTR 0.059%).** Action:
   `review_for_ctr_fix`. Why: biggest impression volume in the queue, CTR well under the
   4-10 median (0.21%). What would make it wrong: if the query mix behind these impressions
   is mostly navigational/branded (users already know the brand and skip the snippet), low
   CTR here is normal and a title rewrite won't move it.
4. **client 73cda7b4, pos 4.6 (tier 4-10), 58,553 impr, 1 click.** Action:
   `review_for_ctr_fix`. Why: same near-zero-click pattern as #2, second occurrence for this
   client. What would make it wrong: same tracking-gap concern as #2 -- two near-zero-click,
   high-impression rows from the same client is a pattern worth checking at the client/tag
   level before treating either as a content problem.
5. **client 23a62021, pos 9.0 (tier 4-10), 55,680 impr, 15 clicks (CTR 0.027%).** Action:
   `review_for_ctr_fix`. Why: sits right at the bottom of page one (position 9), CTR far
   under the tier median. What would make it wrong: if position is volatile day-to-day and
   this 15-day average masks a page that only just cracked the top 10 near the end of the
   window -- a page new to page one hasn't had time to earn clicks yet.
6. **client 62f4a7e6, pos 6.4 (tier 4-10), 49,314 impr, 2 clicks.** Action:
   `review_for_ctr_fix`. Why: third row from this same client in the top 10. What would make
   it wrong: at this point the pattern itself is the finding -- see the weak-pick note below
   about one client dominating the queue.
7. **client 20259bd6, pos 18.3 (tier 11-20), 70,169 impr, 29 clicks (CTR 0.041%).** Action:
   `review_for_ctr_fix`. Why: large impression volume for a page near the bottom of page 2,
   CTR well under its tier's median (0.186%). What would make it wrong: position 18 is close
   to falling off page 2 entirely -- if it's about to drop further, a CTR fix is wasted effort
   compared to a ranking-focused refresh instead.
8. **client 62f4a7e6, pos 9.5 (tier 4-10), 49,619 impr, 8 clicks (CTR 0.016%).** Action:
   `review_for_ctr_fix`. Why: fourth row from this client. What would make it wrong: same
   client-concentration concern as #6.
9. **client 62f4a7e6, pos 4.0 (tier 4-10), 52,378 impr, 18 clicks (CTR 0.034%).** Action:
   `review_for_ctr_fix`. Why: good position (top 5), still far under tier-median CTR. What
   would make it wrong: fifth row from this client -- by now this page may be a perfectly
   legitimate pick, but it can't be told apart from a client-level anomaly using this rule
   alone.
10. **client 62f4a7e6, pos 5.2 (tier 4-10), 46,199 impr, 11 clicks (CTR 0.024%).** Action:
    `review_for_ctr_fix`. Why: sixth row from this client, same story. What would make it
    wrong: same concentration concern -- see section 4.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [6]:
client_counts = queue.head(10)["client_hash_id"].value_counts()
print("client_hash_id counts within the top 10:")
print(client_counts)

zero_or_near_zero = queue.head(10)[queue.head(10)["clicks_first_half"] <= 1]
print(f"\ntop-10 rows with <=1 click despite high impressions: {len(zero_or_near_zero)}")
print(zero_or_near_zero[["client_hash_id", "impressions_first_half", "clicks_first_half"]])

client_hash_id counts within the top 10:
client_hash_id
client_62f4a7e64f5e0096    6
client_73cda7b4e4f265ea    2
client_23a62021009f63c4    1
client_20259bd6705d81d4    1
Name: count, dtype: int64

top-10 rows with <=1 click despite high impressions: 2
            client_hash_id  impressions_first_half  clicks_first_half
1  client_73cda7b4e4f265ea             83,772.0000             0.0000
3  client_73cda7b4e4f265ea             58,553.0000             1.0000


**Weak picks, named:**

1. **One client dominates the top 10.** `client_62f4a7e6...` supplies 6 of the top 10 rows
   (#1, #3, #6, #8, #9, #10); a second client supplies 2 more. Because the score is an
   absolute "missed clicks" count, it structurally favors whichever client happens to run the
   highest-traffic pages -- a global top-10 isn't really "the 10 best opportunities across the
   lane," it's closer to "client 62f4a7e6's biggest pages." A v2 baseline should rank
   *within* client first, or normalize the score by each client's total impression volume,
   before combining into one global queue.
2. **Near-zero-click, high-impression rows may not be content problems at all.** Rows #2 and
   #4 have 0 and 1 click on 58k-84k impressions -- CTR that low usually means a tracking or
   indexing artifact (a redirect, a canonical mismatch, GSC counting impressions against the
   wrong URL) rather than a snippet a copywriter can fix. Flagging these for `review_for_ctr_fix`
   without a technical check first would waste review time on the wrong team.

**Leakage check (same lesson as ML-04):**
- No `is_declining_label` or any second-half (2026-03-16 to 2026-03-31) value anywhere in the
  score, reason code, or action label -- everything is computed from the first-half window
  only.
- `content_updated_date` was tested and rejected specifically because it postdates the
  decision point for 82.7% of rows (printed in section 1) -- using it would have been a
  future-window leak.
- `content_age_days` (signal 1) was tested and found MIXED/not usable -- it is **not** in the
  score, so a rejected signal never quietly entered the rule anyway.
- No product-decision flags (`is_published`/`is_deleted` are used only as a data-quality
  filter, never as a scoring input) and no client names, URLs, or raw queries appear anywhere
  in this notebook.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled -- markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime -> Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` -- then submit your repo URL on the card. Done.